# Data Cleaning

### dataset : usedCars.csv

In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [9]:
# Load dataset
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "usedCars.csv"
CLEAN_DATA_PATH = DATA_DIR / "cleaned_data.csv"
print("Root Directory: ", PROJECT_ROOT)
print("Data Directory: ", DATA_DIR)
print("Raw Data Path: ", RAW_DATA_PATH)
print("Cleaned Data Path: ", CLEAN_DATA_PATH)

Root Directory:  D:\study_material\ProjectFinal\car_price
Data Directory:  D:\study_material\ProjectFinal\car_price\data
Raw Data Path:  D:\study_material\ProjectFinal\car_price\data\usedCars.csv
Cleaned Data Path:  D:\study_material\ProjectFinal\car_price\data\cleaned_data.csv


In [10]:
# Load data
data = pd.read_csv(RAW_DATA_PATH)

In [11]:
# Data back-up
df = data.copy()

In [12]:
# Meta data
print("No. of rows: ", df.shape[0])
print("No. of columns: ", df.shape[1])
print("All columns: \n", df.columns.to_list())
print("Duplicate Count: ", df.duplicated().sum())
print("Null Values: \n", df.isnull().sum())
print("\nInfo:\n")
df.info()

No. of rows:  1064
No. of columns:  19
All columns: 
 ['Id', 'Company', 'Model', 'Variant', 'FuelType', 'Colour', 'Kilometer', 'BodyStyle', 'TransmissionType', 'ManufactureDate', 'ModelYear', 'CngKit', 'Owner', 'DealerState', 'DealerName', 'City', 'Warranty', 'QualityScore', 'Price']
Duplicate Count:  0
Null Values: 
 Id                     0
Company                0
Model                  0
Variant                0
FuelType               1
Colour                 0
Kilometer              0
BodyStyle              0
TransmissionType     714
ManufactureDate        0
ModelYear              0
CngKit              1042
Owner                  0
DealerState            0
DealerName             0
City                   0
Warranty               0
QualityScore           0
Price                  0
dtype: int64

Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1064 entries, 0 to 1063
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            -------

In [13]:
df.head(5)

,Id,Company,Model,Variant,FuelType,Colour,Kilometer,BodyStyle,TransmissionType,ManufactureDate,ModelYear,CngKit,Owner,DealerState,DealerName,City,Warranty,QualityScore,Price
0,555675,MARUTI SUZUKI,CELERIO(2017-2019),1.0 ZXI AMT O,PETROL,Silver,33197,HATCHBACK,NaN,01-02-2018,2018,NaN,1st Owner,Karnataka,Top Gear Cars,Bangalore,1,7.8,5.75 Lakhs
1,556383,MARUTI SUZUKI,ALTO,LXI,PETROL,Red,10322,HATCHBACK,Manual,01-03-2021,2021,NaN,1st Owner,Karnataka,Renew 4 u Automobiles PVT Ltd,Bangalore,1,8.3,4.35 Lakhs
2,556422,HYUNDAI,GRAND I10,1.2 KAPPA ASTA,PETROL,Grey,37889,HATCHBACK,Manual,01-03-2015,2015,NaN,1st Owner,Karnataka,Anant Cars Auto Pvt Ltd,Bangalore,1,7.9,4.7 Lakhs
3,556771,TATA,NEXON,XT PLUS,PETROL,A Blue,13106,HATCHBACK,NaN,01-08-2020,2020,NaN,1st Owner,Karnataka,Adeep Motors,Bangalore,1,8.1,9.9 Lakhs
4,559619,FORD,FIGO,EXI DURATORQ 1.4,DIESEL,Silver,104614,HATCHBACK,Manual,01-11-2010,2010,NaN,2nd Owner,Karnataka,Zippy Automart,Bangalore,0,7.5,2.7 Lakhs


In [14]:
# Drop irrelevant columns
df.drop(columns = ["Id", "CngKit", "DealerState", "DealerName","City","TransmissionType","Variant","Colour","BodyStyle"], axis = 1, inplace = True)
print("Final list of columns: \n", df.columns.to_list())

Final list of columns: 
 ['Company', 'Model', 'FuelType', 'Kilometer', 'ManufactureDate', 'ModelYear', 'Owner', 'Warranty', 'QualityScore', 'Price']


In [15]:
df["ManufactureDate"] = pd.to_datetime(df["ManufactureDate"], errors="coerce")

In [16]:
# Extract year
df["ManufactureYear"] = df["ManufactureDate"].dt.year

# Calculate car age in years
current_year = pd.Timestamp.today().year
df["CarAge"] = current_year - df["ManufactureYear"]

In [17]:
df.head()

,Company,Model,FuelType,Kilometer,ManufactureDate,ModelYear,Owner,Warranty,QualityScore,Price,ManufactureYear,CarAge
0,MARUTI SUZUKI,CELERIO(2017-2019),PETROL,33197,2018-01-02,2018,1st Owner,1,7.8,5.75 Lakhs,2018,8
1,MARUTI SUZUKI,ALTO,PETROL,10322,2021-01-03,2021,1st Owner,1,8.3,4.35 Lakhs,2021,5
2,HYUNDAI,GRAND I10,PETROL,37889,2015-01-03,2015,1st Owner,1,7.9,4.7 Lakhs,2015,11
3,TATA,NEXON,PETROL,13106,2020-01-08,2020,1st Owner,1,8.1,9.9 Lakhs,2020,6
4,FORD,FIGO,DIESEL,104614,2010-01-11,2010,2nd Owner,0,7.5,2.7 Lakhs,2010,16


In [18]:
df.drop(columns = ["ManufactureDate", "ManufactureYear"], axis = 1, inplace = True)
print("Final list of columns: \n", df.columns.to_list())

Final list of columns: 
 ['Company', 'Model', 'FuelType', 'Kilometer', 'ModelYear', 'Owner', 'Warranty', 'QualityScore', 'Price', 'CarAge']


In [19]:
# Step 1: Remove the word 'Lakhs' and strip spaces
df["Price"] = df["Price"].str.replace("Lakhs", "", regex=False).str.strip()

# Step 2: Remove commas if any (like '5,75')
df["Price"] = df["Price"].str.replace(",", "", regex=False)

# Step 3: Convert to float safely
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

# Step 4: Convert Lakhs to Rupees (optional)
df["Price"] = df["Price"] * 100000

# Step 5: Print the cleaned values
print(df["Price"])

0       575000.0
1       435000.0
2       470000.0
3       990000.0
4       270000.0
          ...   
1059    715000.0
1060    500000.0
1061    575000.0
1062    771000.0
1063    499000.0
Name: Price, Length: 1064, dtype: float64


In [20]:
# Save c;lean data
df.to_csv(CLEAN_DATA_PATH, index = False)